# 03 - Lineshape Analysis and Fitting

`epyr.lineshapes` provides the standard EPR profiles, all area-normalized, with
support for derivative order and phase. `epyr.lineshapes.fitting` fits these
profiles to data.

Note: `voigtian` takes `widths` as a 2-tuple `(gaussian_fwhm, lorentzian_fwhm)`,
whereas `gaussian`, `lorentzian`, and `pseudo_voigt` take a single `width`.

In [ ]:
%matplotlib inline
import warnings
warnings.filterwarnings("ignore")  # keep tutorial output readable

import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

import epyr

DATA = Path("..") / "data"   # example datasets, relative to this notebook
print("EPyR Tools version:", epyr.__version__)

In [ ]:
from epyr.lineshapes import gaussian, lorentzian, voigtian, pseudo_voigt, Lineshape

## The four profiles

In [ ]:
B = np.linspace(-10, 10, 1000)   # mT

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(B, gaussian(B, 0, 4.0), label="Gaussian")
ax.plot(B, lorentzian(B, 0, 4.0), label="Lorentzian")
ax.plot(B, voigtian(B, 0, (3.0, 3.0)), label="Voigt")
ax.plot(B, pseudo_voigt(B, 0, 4.0, eta=0.5), "--", label="pseudo-Voigt")
ax.legend(); ax.set_xlabel("Field offset / mT"); ax.set_title("Area-normalized lineshapes")
plt.show()

## Derivatives

Field-modulated CW EPR records the first derivative. Pass `derivative=0, 1, 2`.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
for order in (0, 1, 2):
    ax.plot(B, gaussian(B, 0, 4.0, derivative=order), label=f"derivative {order}")
ax.legend(); ax.set_xlabel("Field offset / mT"); ax.set_title("Gaussian derivatives")
plt.show()

## The `Lineshape` class

A reusable object with fixed shape parameters, called like a function.

In [ ]:
shape = Lineshape("pseudo_voigt", width=4.0, alpha=0.6)
y_shape = shape(B, center=0.0)
print("peak value:", y_shape.max())

## Fitting a real first-derivative spectrum

`fit_epr_signal` fits one model and returns a `FitResult` (parameters, errors,
R^2, reduced chi^2). With `plot=True` it draws the fit and residuals.

In [ ]:
from epyr.lineshapes.fitting import fit_epr_signal, fit_multiple_shapes

xc, yc, pc, _ = epyr.eprload(DATA / "CuSO4_001.par", plot_if_possible=False)
result = fit_epr_signal(xc, yc, "lorentzian", derivative=1, plot=True)
plt.show()
print(result.summary())

## Comparing models

`fit_multiple_shapes` fits several profiles and returns a dict
`{name: FitResult}`, so you can compare goodness of fit.

In [ ]:
# Synthetic pseudo-Voigt peak with noise
xs = np.linspace(3350, 3550, 600)
ys = 1.0 * pseudo_voigt(xs, 3450, 18.0, eta=0.4) + np.random.normal(0, 2e-5, xs.size)

results = fit_multiple_shapes(xs, ys, plot=True)
plt.show()
for name, r in results.items():
    print(f"{name:14s} R^2 = {r.r_squared:.5f}  success={r.success}")

## Summary

- Lineshapes are area-normalized and support `derivative` and `phase`.
- `voigtian` uses a `(gaussian_fwhm, lorentzian_fwhm)` tuple.
- `fit_epr_signal` fits one model; `fit_multiple_shapes` compares several.